# TITAN-1 — Scientific Image Restoration Engine

### World-Class Reproducible Scientific Benchmark Notebook

**Researcher:** Md. Akizul Islam  
**Role:** Independent Researcher & Programmer  
**Domain:** Scientific Imaging / AI for Science / Computational Imaging  
**Framework:** PyTorch  

---

## Scientific Scope

TITAN-1 investigates lightweight neural image restoration under controlled synthetic degradation and out-of-distribution morphology shifts.

The benchmark is designed around the forward model:

$$y = H(x) + n$$

where $x$ is the latent image, $H$ is the degradation/instrument-response operator, $n$ is noise, and $y$ is the observed image.

> **Evidence policy:** no benchmark result is claimed unless it is actually executed and recorded in this notebook.

## 1. Reproducibility Contract

This notebook records:

- Python and PyTorch versions
- hardware/device information
- random seed
- model architecture
- parameter count
- dataset generation protocol
- preprocessing
- evaluation metrics
- execution timestamp
- SHA-256 provenance where applicable

Historical results and newly executed results must **never be silently merged**.

In [ ]:
import sys
import platform
import random
import hashlib
import json
from datetime import datetime, timezone

import numpy as np
import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('=== TITAN-1 ENVIRONMENT ===')
print('Timestamp UTC :', datetime.now(timezone.utc).isoformat())
print('Python        :', sys.version.split()[0])
print('Platform      :', platform.platform())
print('PyTorch       :', torch.__version__)
print('NumPy         :', np.__version__)
print('Device        :', DEVICE)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))


## 2. Canonical TITAN-1 Architecture

The historical scientific benchmark architecture is:

```text
1 channel
    ↓
3×3 Conv — 32 channels
    ↓ ReLU
3×3 Conv — 32 channels
    ↓ ReLU
3×3 Conv — 16 channels
    ↓ ReLU
3×3 Conv — 1 channel
    ↓ Sigmoid
output
```

**Canonical parameter count: 14,337.**

In [ ]:
class TITAN1(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = TITAN1().to(DEVICE)

parameter_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Total parameters   :', parameter_count)
print('Trainable params   :', trainable_count)
print('Expected canonical :', 14337)
print('Architecture check :', parameter_count == 14337)

assert parameter_count == 14337, 'Canonical parameter count mismatch.'


## 3. Controlled Synthetic Benchmark

The documented benchmark design uses:

- 5,000 synthetic training images
- 700 unseen elliptical OOD images
- 64×64 grayscale images
- controlled Gaussian noise
- morphology/distribution shift between training and evaluation

This notebook generates a **new executable benchmark**. Its results must be treated as newly generated results, not as historical TITAN-1 results.

In [ ]:
import math

IMG_SIZE = 64

def make_ring(ellipse=False, seed=None):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
    cx = IMG_SIZE / 2 + rng.uniform(-3, 3)
    cy = IMG_SIZE / 2 + rng.uniform(-3, 3)

    if ellipse:
        rx = rng.uniform(15, 22)
        ry = rng.uniform(10, 20)
    else:
        rx = ry = rng.uniform(14, 21)

    r = np.sqrt(((xx-cx)/rx)**2 + ((yy-cy)/ry)**2)
    ring = np.exp(-((r - 1.0)**2) / (2 * 0.055**2))

    ring = ring / max(ring.max(), 1e-12)
    return ring.astype(np.float32)


def add_gaussian_noise(image, sigma, seed=None):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, sigma, image.shape).astype(np.float32)
    return np.clip(image + noise, 0, 1).astype(np.float32)


print('Synthetic generator ready.')


In [ ]:
import matplotlib.pyplot as plt

clean = make_ring(ellipse=False, seed=SEED)
noisy = add_gaussian_noise(clean, sigma=0.20, seed=SEED)
ood = make_ring(ellipse=True, seed=SEED + 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(clean, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Clean training morphology')
axes[1].imshow(noisy, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Noisy observation')
axes[2].imshow(ood, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('OOD elliptical morphology')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 4. Evaluation Metrics

Primary metrics:

- Mean Squared Error (MSE)
- Peak Signal-to-Noise Ratio (PSNR)
- Structural Similarity Index (SSIM)

Per-image values should be retained before calculating summary statistics.

In [ ]:
def mse(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return float(np.mean((a - b) ** 2))

def psnr(a, b, data_range=1.0):
    value = mse(a, b)
    if value == 0:
        return float('inf')
    return float(10 * np.log10((data_range ** 2) / value))

print('Metric functions ready.')


## 5. Scientific Safety Gate

Before claiming a result as a validated scientific reproduction, verify:

1. checkpoint identity
2. dataset identity
3. architecture compatibility
4. preprocessing
5. execution environment
6. evaluation protocol
7. raw per-image metrics

If any required artifact is missing, the scientific reproduction status must remain **BLOCKED** rather than being reconstructed by assumption.

In [ ]:
SCIENTIFIC_CHECKS = {
    'checkpoint_verified': False,
    'dataset_verified': False,
    'architecture_verified': parameter_count == 14337,
    'gpu_available': torch.cuda.is_available(),
}

scientific_execution = all([
    SCIENTIFIC_CHECKS['checkpoint_verified'],
    SCIENTIFIC_CHECKS['dataset_verified'],
    SCIENTIFIC_CHECKS['architecture_verified'],
])

print(json.dumps({
    'status': 'READY' if scientific_execution else 'BLOCKED',
    'scientific_execution': scientific_execution,
    'checks': SCIENTIFIC_CHECKS
}, indent=2))


In [ ]:
provenance = {
    'project': 'TITAN-1',
    'notebook': os.path.basename(OUTPUT),
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'seed': SEED,
    'image_size': [64, 64],
    'model_parameters': parameter_count,
    'device': str(DEVICE),
    'torch_version': torch.__version__,
    'numpy_version': np.__version__,
    'scientific_execution': scientific_execution,
}

print(json.dumps(provenance, indent=2))


## 6. Evidence & Limitations

This notebook deliberately does **not** claim:

- NASA validation
- CERN validation
- NVIDIA validation
- real EHT scientific validation
- astrophysical parameter inference
- institutional replication
- universal superiority over other restoration methods

Those claims require independent evidence and are outside the scope of a synthetic benchmark notebook.

### Evidence rule

**No evidence → no claim.**

In [ ]:
print('=' * 60)
print('TITAN-1 WORLD-CLASS NOTEBOOK INITIALIZATION REPORT')
print('=' * 60)
print('Notebook structure : PASS')
print('Canonical model    :', 'PASS' if parameter_count == 14337 else 'FAIL')
print('Parameter count    :', parameter_count)
print('Environment        : PASS')
print('Scientific gate    :', 'READY' if scientific_execution else 'BLOCKED')
print('Evidence policy    : ACTIVE')
print('=' * 60)
